## Bước 1: Tiền xử lý dữ liệu
#### Import các thư viện cần thiết cho việc xử lý dữ liệu

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

from pycocotools.coco import COCO
from PIL import Image

print('Finish importing libraries')

#### Chuyển đổi annotations COCO format thành mask images để chuẩn bị dữ liệu cho việc training U-Net segmentation model

In [ ]:
# Đường dẫn dataset gốc
BASE_DIR = '/kaggle/input/brain-tumor-image-dataset-semantic-segmentation'
# Thư mục output
MASKS_DIR = "/kaggle/working/masks"
os.makedirs(MASKS_DIR, exist_ok=True)

#### Hàm sinh mask cho 3 tập train, test và valid

In [ ]:
def generate_masks(split):
    """
    split: "train", "test", "valid"
    """
    input_dir = os.path.join(BASE_DIR, split)
    ann_file = os.path.join(input_dir, "_annotations.coco.json")
    masks_dir = os.path.join(MASKS_DIR, split)
    os.makedirs(masks_dir, exist_ok=True)

    coco = COCO(ann_file)
    img_ids = coco.getImgIds()

    print(f"Processing {split}: {len(img_ids)} images")

    for img_id in img_ids:
        img_info = coco.loadImgs(img_id)[0]
        img_name = img_info['file_name']
        height, width = img_info["height"], img_info["width"]

        # Mask trắng đen
        mask = np.zeros((height, width), dtype=np.uint8)

        # Annotation của ảnh 
        ann_ids = coco.getAnnIds(imgIds=img_id)
        anns = coco.loadAnns(ann_ids)

        for ann in anns:
            m = coco.annToMask(ann)
            mask = np.maximum(mask, m * 255)  # cộng dồn các tumor

        # Lưu mask
        mask_name = os.path.splitext(img_name)[0] + "_mask.png"
        mask_path = os.path.join(masks_dir, mask_name)
        Image.fromarray(mask).save(mask_path)

    print(f"Done {split}, masks saved to {masks_dir}")

In [ ]:
# Tạo mask cho cả 3 folder
for split in ["train", "test", "valid"]:
    generate_masks(split)

#### Hàm show ảnh từ dataset và mask tương ứng của nó

In [ ]:
def show_random_pair(split="train", overlay=True):
    """
    split: "train", "test", "valid"
    overlay: nếu True thì hiển thị thêm ảnh chồng (MRI + mask)
    """
    img_dir = os.path.join(BASE_DIR, split)
    mask_dir = os.path.join(MASKS_DIR, split)

    # Lấy danh sách ảnh gốc
    img_files = [f for f in os.listdir(img_dir) if f.endswith(".jpg") or f.endswith(".png")]
    
    # Random chọn 1 ảnh
    img_name = random.choice(img_files)

    # Đọc ảnh gốc
    img_path = os.path.join(img_dir, img_name)
    img = Image.open(img_path).convert("RGB")
    img_np = np.array(img)

    # Đọc mask tương ứng
    mask_name = os.path.splitext(img_name)[0] + "_mask.png"
    mask_path = os.path.join(mask_dir, mask_name)
    mask = Image.open(mask_path).convert("L")
    mask_np = np.array(mask)

    # Tạo overlay (mask màu đỏ trong suốt)
    overlay_img = img_np.copy()
    overlay_img[mask_np > 0] = [255, 0, 0]  # tô đỏ vùng tumor
    blended = (0.6 * img_np + 0.4 * overlay_img).astype(np.uint8)

    # Vẽ ảnh
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3 if overlay else 2, 1)
    plt.imshow(img)
    plt.title("Image")
    plt.axis("off")

    plt.subplot(1, 3 if overlay else 2, 2)
    plt.imshow(mask, cmap="gray")
    plt.title("Mask")
    plt.axis("off")

    if overlay:
        plt.subplot(1, 3, 3)
        plt.imshow(blended)
        plt.title("Overlay")
        plt.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
# Ví dụ: random 5 cặp trong folder train + overlay
for _ in range(5):
    show_random_pair(split="train", overlay=True)

## Bước 2: Tạo và huấn luyện mô hình Unet

#### Tải thư viện segmentation_models chứa các mô hình đã được build sẵn như Unet, Unet++

In [ ]:
!pip install -U segmentation-models-pytorch

In [ ]:
import cv2

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torch.amp import autocast, GradScaler

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp

print('Finish importing libraries')

In [ ]:
# Các siêu tham số và hằng số cho việc huấn luyện
IMG_SIZE = 256
BATCH_SIZE = 64
EPOCHS = 80
LR = 1e-4
NUM_WORKERS = 4
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Set seed
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [ ]:
# Utils: Transform
train_transform = A.Compose([
    # Biến đổi hình học (áp dụng cho cả ảnh + mask)
    A.Resize(256, 256), 
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Affine(scale=(0.9, 1.1), translate_percent=(0.1, 0.1), 
             rotate=(-15, 15), p=0.5),
    A.ElasticTransform(p=0.3),
    A.GridDistortion(p=0.3),

    # Biến đổi màu sắc / nhiễu (chỉ áp dụng cho ảnh, không ảnh hưởng mask)
    A.RandomBrightnessContrast(p=0.5),
    A.ColorJitter(p=0.5),
    A.GaussNoise(p=0.2),

    # Chuẩn hóa + chuyển tensor
    A.Normalize(mean=(0.485, 0.456, 0.406), 
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(mean=(0.485, 0.456, 0.406), 
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

test_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(mean=(0.485, 0.456, 0.406), 
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

test_robust_transform = A.Compose([
    A.Resize(256, 256),
    A.RandomBrightnessContrast(p=0.8),   # sáng/tối khác nhau
    A.GaussNoise(p=0.8),  # nhiễu Gaussian
    A.MotionBlur(p=0.5),                 # ảnh bị mờ do rung tay
    A.GridDistortion(p=0.5),             # biến dạng nhẹ
    A.ColorJitter(p=0.5),                # thay đổi màu sắc
    A.Normalize(mean=(0.485, 0.456, 0.406), 
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])


In [ ]:
# Khởi tạo Dataset 
class TumorSegDataset(Dataset):
    def __init__(self, split="train", transform=None):
        self.split = split
        self.img_dir = os.path.join(BASE_DIR, split)
        self.mask_dir = os.path.join(MASKS_DIR, split)

        self.img_files = [f for f in os.listdir(self.img_dir)
                          if f.lower().endswith((".jpg", ".png")) and not f.startswith("_")]
        self.img_files.sort()
        self.transform = transform

        # kiểm tra mask tồn tại
        missing = []
        for f in self.img_files:
            mask_name = os.path.splitext(f)[0] + "_mask.png"
            if not os.path.exists(os.path.join(self.mask_dir, mask_name)):
                missing.append(mask_name)
        if missing:
            print(f"[WARN] Missing {len(missing)} masks in {self.mask_dir}. Example: {missing[:5]}")

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        f = self.img_files[idx]
        img_path = os.path.join(self.img_dir, f)
        mask_path = os.path.join(self.mask_dir, os.path.splitext(f)[0] + "_mask.png")

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, 0)
        mask = (mask > 127).astype("float32")
        
        # Áp dụng augmentation cho cả ảnh và mask
        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented["image"]
            mask = augmented["mask"]

        # Đưa mask về tensor đúng kiểu
        mask = mask.clone().detach().unsqueeze(0)
        return img, mask

In [ ]:
# Tạo dataset cho train, test và valid
train_dataset = TumorSegDataset("train", transform=train_transform)
val_dataset = TumorSegDataset("valid", transform=val_transform)
test_dataset = TumorSegDataset("test", transform=test_transform)
test_robust_dataset = TumorSegDataset("test", transform=test_robust_transform)

# Tạo dataloader cho train, test và valid
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_robust_loader = DataLoader(test_robust_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

#### Khởi tạo kiến trúc và huấn luyện mô hình Unet

In [ ]:
import gc

# Handling CUDA OOM
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Model Unet từ smp
model = smp.UnetPlusPlus(
    encoder_name='resnet34',     # backbone resnet34
    encoder_weights='imagenet',  # pretrained imagenet,
    in_channels=3,               # RGB
    classes=1,                   # binary mask           
).to(DEVICE)

# Loss và Optimizer
bce_loss = nn.BCEWithLogitsLoss()

def dice_loss(pred, target, epsilon=1e-7):
    pred = torch.sigmoid(pred)
    pred = pred.contiguous().view(-1)
    target = target.contiguous().view(-1)

    num = 2. * (pred * target).sum()
    den = pred.sum() + target.sum()

    dice = (num + epsilon) / (den + epsilon)
    return 1 - dice

def combined_loss(pred, target):
    return 0.5 * bce_loss(pred, target) + 0.5 * dice_loss(pred, target)

def focal_tversky_loss(pred, target, alpha=0.7, beta=0.3, gamma=4/3, smooth=1e-7):
    pred = torch.sigmoid(pred)
    TP = (pred * target).sum()
    FP = ((1 - target) * pred).sum()
    FN = (target * (1 - pred)).sum()
    Tversky = (TP + smooth) / (TP + alpha*FP + beta*FN + smooth)
    return (1 - Tversky) ** gamma


In [ ]:
# Optimizer và lr_scheduler
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)

lr_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=10
)

scaler = GradScaler('cuda')

In [ ]:
print(torch.cuda.memory_summary())

In [ ]:
# Metrics
def iou_score(pred, target, threshold=0.5, epsilon=1e-7):
    pred = (torch.sigmoid(pred) > threshold).float()
    inter = (pred * target).sum()
    union = pred.sum() + target.sum() - inter
    return (inter + epsilon)/(union + epsilon)

def dice_score(pred, target, threshold=0.5, epsilon=1e-7):
    pred = (torch.sigmoid(pred) > threshold).float()
    inter = (pred * target).sum()
    return (2 * inter + epsilon) / (pred.sum() + target.sum() + epsilon)

In [ ]:
# Train model
def train_one_epoch(loader):
    model.train()
    total_loss = 0
    for imgs,masks in loader:
        imgs,masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        
        with autocast('cuda'):
            logits = model(imgs)
            loss = combined_loss(logits, masks)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    
        total_loss += loss.item() * imgs.size(0)
    return total_loss / len(loader.dataset)

In [ ]:
# Validate model
def validate(loader):
    model.eval()
    total_loss = 0
    total_iou = 0
    total_dice = 0
    with torch.no_grad():
        for imgs,masks in loader:
            imgs,masks = imgs.to(DEVICE), masks.to(DEVICE)
            logits = model(imgs)
            loss = combined_loss(logits, masks)
            total_loss += loss.item() * imgs.size(0)
            total_iou += iou_score(logits,masks).item() * imgs.size(0)
            total_dice += dice_score(logits,masks).item() * imgs.size(0)
    n = len(loader.dataset)
    return total_loss/n, total_iou/n, total_dice/n

In [ ]:
# Training
best_dice = 0
best_iou = 0
patience = 10
counter = 0
real_epochs = EPOCHS
train_losses, val_losses, val_ious, val_dices = [], [], [], []

for epoch in range(EPOCHS):
    tr_loss = train_one_epoch(train_loader)
    v_loss, v_iou, v_dice = validate(val_loader)

    # lr_scheduler
    lr_scheduler.step(v_dice)
    
    train_losses.append(tr_loss)
    val_losses.append(v_loss)
    val_ious.append(v_iou)
    val_dices.append(v_dice)
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Train loss {tr_loss:.4f} | "
          f"Val loss {v_loss:.4f} | Val IoU {v_iou:.4f} | Val Dice {v_dice:.4f}")

    # Save checkpoint
    if v_dice > best_dice:
        best_dice = v_dice
        best_iou = v_iou
        torch.save(model.state_dict(), "/kaggle/working/unet_best.pth")
        counter = 0
    else:
        counter += 1

    # Early stopping
    # if counter >= patience:
    #     real_epochs = epoch + 1
    #     print("Early stopping triggered!")
    #     break
    
print(f"Best dice: {best_dice:.4f}")
print(f"Best IoU: {best_iou:.4f}")

## Bước 3: Visualize quá trình huấn luyện và hiển thị dự đoán của mô hình

In [ ]:
# Visualize quá trình huấn luyện
def plot_training(train_losses, val_losses, val_dices, val_ious):
    epochs = [epoch for epoch in range(1, real_epochs + 1)]
    plt.figure(figsize=(18,6))

    plt.subplot(1,3,1)
    plt.plot(epochs, train_losses, 'bo-', label='Train Loss')
    plt.plot(epochs, val_losses, 'ro-', label='Val Loss')
    plt.title("Loss over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid()

    plt.subplot(1,3,2)
    plt.plot(epochs, val_dices, 'go-', label='Val Dice')
    plt.title("Validation Dice")
    plt.xlabel("Epoch")
    plt.ylabel("Dice")
    plt.legend()
    plt.grid()

    plt.subplot(1,3,3)
    plt.plot(epochs, val_ious, 'mo-', label='Val IoU')
    plt.title("Validation IoU")
    plt.xlabel("Epoch")
    plt.ylabel("IoU")
    plt.legend()
    plt.grid()

    plt.tight_layout()
    # Save figure
    plt.savefig("/kaggle/working/train_fig.png")
    plt.show()

    
    
# Visualize preformance của model trên tập test
def plot_test(test_losses, test_dices, test_ious):
    epochs = [epoch for epoch in range(1, real_epochs + 1)]
    plt.figure(figsize=(18,6))

    plt.subplot(1,3,1)
    plt.plot(epochs, test_losses, 'bo-', label='Test Loss')
    plt.title("Test Loss over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid()

    plt.subplot(1,3,2)
    plt.plot(epochs, test_dices, 'go-', label='Test Dice')
    plt.title("Test Dice")
    plt.xlabel("Epoch")
    plt.ylabel("Dice")
    plt.legend()
    plt.grid()

    plt.subplot(1,3,3)
    plt.plot(epochs, test_ious, 'mo-', label='Test IoU')
    plt.title("Test IoU")
    plt.xlabel("Epoch")
    plt.ylabel("IoU")
    plt.legend()
    plt.grid()

    plt.tight_layout()
    # Save figure
    plt.savefig("/kaggle/working/test_fig.png")
    plt.show()
    
# Chạy hàm visualize
plot_training(train_losses, val_losses, val_dices, val_ious)
# plot_test(test_losses, test_dices, test_ious)

In [ ]:
def denormalize(image, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
    """Denormalize ảnh tensor (C,H,W) về [0,255]."""
    image = image.cpu().numpy()
    image = image.transpose(1,2,0)  # (H,W,C)
    image = (image * std + mean)    # undo normalization
    image = np.clip(image * 255, 0, 255).astype(np.uint8)
    return image
    
def show_prediction(model, dataset, idx=None, threshold=0.5, save_img=False, output_dir=None):
    """
    Hiển thị ảnh gốc (denormalized) với mask GT (đỏ) và Prediction (xanh).
    """
    # Chọn ảnh
    if idx is None:
        idx = random.randint(0, len(dataset) - 1)
    image, mask = dataset[idx]  # image: Tensor (C,H,W), mask: Tensor

    # Chuẩn bị tensor cho model
    img_tensor = image.unsqueeze(0).to(DEVICE)  # (1,C,H,W)
    
    # Denormalize ảnh để hiển thị
    img_np = denormalize(image)  # (H,W,C)

    # Chuẩn bị mask GT (ép về H,W)
    mask = mask.cpu().numpy()
    mask = np.squeeze(mask)   # (H,W)

    # Predict (ép về H,W)
    with torch.no_grad():
        pred_mask = model.predict(img_tensor)
    pred_mask = (pred_mask.squeeze().cpu().numpy() > threshold).astype(np.uint8)  # (H,W)

    # Copy ảnh gốc để tô màu
    gt = img_np.copy()
    pred = img_np.copy()

    # Tô màu (chỉ vùng mask được thay đổi)
    gt[mask > 0] = [255, 0, 0]       # GT: đỏ
    pred[pred_mask > 0] = [0, 255, 0]  # Prediction: xanh lá
    # Làm mờ vùng được đánh dấu
    gt_blended = (0.6 * img_np + 0.4 * gt).astype(np.uint8)
    pred_blended = (0.6 * img_np + 0.4 * pred).astype(np.uint8)
    
    # Hiển thị
    plt.figure(figsize=(18, 6))
    plt.subplot(1,3,1)
    plt.imshow(img_np)
    plt.title(f"Original with index {idx}")
    plt.axis("off")

    plt.subplot(1,3,2)
    plt.imshow(gt_blended)
    plt.title("Ground Truth")
    plt.axis("off")

    plt.subplot(1,3,3)
    plt.imshow(pred_blended)
    plt.title("Prediction")
    plt.axis("off")

    if save_img and output_dir is not None:
        os.makedirs(output_dir, exist_ok=True)
        save_path = os.path.join(output_dir, f"img{idx}.png")
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()


In [ ]:
for _ in range(5):
    show_prediction(model=model, dataset=test_robust_dataset)

## Bước 4: Load thử pretrained model và dự đoán

In [ ]:
pretrained_model = smp.UnetPlusPlus(
    encoder_name='resnet34',     # backbone resnet34
    encoder_weights='imagenet',  # pretrained imagenet
    in_channels=3,               # RGB
    classes=1,                   # binary mask           
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Load mô hình 
pretrained_model.load_state_dict(torch.load("/kaggle/working/unet_best.pth", map_location=device))
pretrained_model.to(device)
pretrained_model.eval()

In [ ]:
# Show 5 ảnh ground truth và prediction
for _ in range(5):
    show_prediction(model=pretrained_model, dataset=test_robust_dataset)

## Bước 5: Lưu lại ground truth và prediction

In [ ]:
for idx in range(0, len(test_robust_dataset) - 1):
    show_prediction(pretrained_model, test_robust_dataset, idx=idx, threshold=0.5, 
                    save_img=True, output_dir="/kaggle/working/output")
print("Save output successfully!")